# Clase 9 — Encapsulación

---

## El problema que resuelve la encapsulación

Hasta ahora nuestros objetos son completamente abiertos. Cualquiera puede entrar y cambiar lo que quiera:

In [ ]:
class CuentaBancaria:
    def __init__(self, titular, saldo):
        self.titular = titular
        self.saldo   = saldo

cuenta = CuentaBancaria("Ana", 1000)
print(f"Saldo inicial: ${cuenta.saldo}")

# Nada impide hacer esto:
cuenta.saldo = -999999
print(f"Saldo después: ${cuenta.saldo}") 

Saldo inicial: $1000
Saldo después: $-999999


Nadie debería poder poner un saldo negativo directo. Necesitamos **controlar el acceso** a los atributos.

---

### Convención: atributos "privados" con `_`

En Python no existe un privado real como en otros lenguajes, pero hay una **convención**: si un atributo empieza con `_`, es una señal de que **no deberías tocarlo desde afuera**.

```python
self._saldo = saldo    # "oye, esto es interno, no lo toques directamente"
```

Técnicamente sigue siendo accesible, pero es un contrato con quien usa tu clase.

---

### La solución: `@property`

Un `@property` permite **leer un atributo como si fuera público**, pero por dentro pasa por una función que nosotros controlamos.

In [ ]:
class CuentaBancaria:
    def __init__(self, titular, saldo_inicial):
        self.titular = titular
        self._saldo  = saldo_inicial   # atributo privado

    @property
    def saldo(self):                   # getter: se llama al LEER cuenta.saldo
        return self._saldo

cuenta = CuentaBancaria("Ana", 1000)

print(cuenta.saldo)     # funciona igual que antes — se lee como atributo normal
# cuenta.saldo = 500    # esto ahora lanza un error — no tiene setter todavía

### Agregando el setter con validación

El setter se define con `@nombre.setter` y es donde ponemos las **reglas**.

In [6]:
class CuentaBancaria:
    IMPUESTO = 0.19 #Evitemos esta sintaxis por ahora

    def __init__(self, titular, saldo_inicial):
        self.titular = titular
        self._saldo  = 0
        self.saldo   = saldo_inicial   # usa el setter desde el inicio

    @property
    def saldo(self):
        return self._saldo

    @saldo.setter
    def saldo(self, valor):
        if valor < 0:
            print(f"  ❌ Saldo inválido: {valor}. El saldo no puede ser negativo.")
            return
        self._saldo = valor

    @property
    def saldo_neto(self):              # solo lectura: no tiene setter
        return round(self._saldo * (1 - self.IMPUESTO), 2)

    def depositar(self, monto):
        if monto <= 0:
            print("  ❌ El monto a depositar debe ser mayor a 0")
            return
        self.saldo += monto
        print(f"  ✅ Depósito de ${monto}. Nuevo saldo: ${self.saldo}")

    def retirar(self, monto):
        if monto > self._saldo:
            print(f"  ❌ Fondos insuficientes (saldo: ${self._saldo})")
            return
        self.saldo -= monto
        print(f"  ✅ Retiro de ${monto}. Nuevo saldo: ${self.saldo}")

    def __str__(self):
        return f"Cuenta de {self.titular} — saldo: ${self._saldo}"

cuenta = CuentaBancaria("Ana", 1000)
print(cuenta)

cuenta.depositar(500)
cuenta.retirar(200)
cuenta.retirar(9999)       # fondos insuficientes
cuenta.saldo = -500        # intento directo — bloqueado por el setter

print(cuenta)
print(f"Saldo bruto: ${cuenta.saldo}  |  Saldo neto (con impuesto): ${cuenta.saldo_neto}")

Cuenta de Ana — saldo: $1000
  ✅ Depósito de $500. Nuevo saldo: $1500
  ✅ Retiro de $200. Nuevo saldo: $1300
  ❌ Fondos insuficientes (saldo: $1300)
  ❌ Saldo inválido: -500. El saldo no puede ser negativo.
Cuenta de Ana — saldo: $1300
Saldo bruto: $1300  |  Saldo neto (con impuesto): $1053.0


### Propiedades de solo lectura

Si defines `@property` **sin** su setter, el atributo es de **solo lectura**: se puede consultar pero no asignar. Sirve para exponer valores **calculados** a partir de otros atributos — como `saldo_neto` arriba, que se recalcula solo cada vez que se consulta.

---

### Segundo ejemplo: clase `Persona`

Mismo patrón, dominio diferente. Aquí la encapsulación protege la **edad** y calcula si es mayor de edad.

---

### Segundo ejemplo: clase `Persona`

Mismo patrón, dominio diferente. Aquí la encapsulación protege la **edad** y calcula si es mayor de edad.

In [7]:
class Persona:
    def __init__(self, nombre, edad):
        self._nombre = nombre
        self._edad   = 0
        self.edad    = edad     # pasa por el setter

    # --- nombre: solo lectura ---
    @property
    def nombre(self):
        return self._nombre

    # --- edad: lectura y escritura con validación ---
    @property
    def edad(self):
        return self._edad

    @edad.setter
    def edad(self, valor):
        if not isinstance(valor, int) or valor < 0 or valor > 150:
            print(f"  ❌ Edad inválida: {valor}")
            return
        self._edad = valor

    # --- propiedad calculada: sin setter ---
    @property
    def es_mayor(self):
        return self._edad >= 18

    def __str__(self):
        estado = "mayor de edad" if self.es_mayor else "menor de edad"
        return f"{self._nombre}, {self._edad} años ({estado})"


p = Persona("Carlos", 17)
print(p)

p.edad = 18
print(p)

p.edad = -5         # bloqueado
p.edad = "veinte"   # bloqueado
print(p)            # sigue en 18

Carlos, 17 años (menor de edad)
Carlos, 18 años (mayor de edad)
  ❌ Edad inválida: -5
  ❌ Edad inválida: veinte
Carlos, 18 años (mayor de edad)


### Resumen

| Qué | Cómo | Para qué |
|---|---|---|
| Atributo privado | `self._attr` | Señal de "no tocar desde afuera" |
| Leer con control | `@property` | Getter — lógica al acceder |
| Escribir con validación | `@attr.setter` | Setter — reglas al asignar |
| Solo lectura | `@property` sin setter | Valor calculado o protegido |

---

## Ejercicios

### 🟢 Ejercicio 1 — Termómetro *(fácil)*

Crea una clase `Termometro` que:

- Almacene la temperatura internamente en **Celsius** (`_celsius`)
- Tenga una property `celsius` con setter que **no permita bajar de -273.15** (cero absoluto)
- Tenga una property **de solo lectura** `fahrenheit` que retorne la conversión (`C × 9/5 + 32`)
- Tenga una property **de solo lectura** `estado` que retorne:
  - `"congelando"` si está bajo 0°C
  - `"normal"` entre 0 y 100°C
  - `"hirviendo"` sobre 100°C

**Prueba:** crea un termómetro, cambia su temperatura varias veces y observa cómo cambia `fahrenheit` y `estado` automáticamente. Intenta poner -300°C.

In [ ]:
# Ejercicio 1



---

### 🟡 Ejercicio 2 — Producto de tienda *(medio)*

Crea una clase `Producto` que represente un artículo en una tienda:

- Atributos privados: `_nombre`, `_precio`, `_stock`
- `nombre`: solo lectura (no se puede cambiar el nombre de un producto)
- `precio`: con setter que rechace valores negativos o cero
- `stock`: con setter que rechace valores negativos
- Método `vender(cantidad)`: descuenta del stock. Rechaza si no hay suficiente stock o si `cantidad <= 0`.
- Método `reabastecer(cantidad)`: suma al stock. Rechaza cantidades negativas.
- Property de solo lectura `valor_total`: retorna `precio × stock` (cuánto vale el inventario)

**Prueba:** crea un producto, vende algunas unidades, intenta vender más de lo que hay, reabastece y muestra el valor total del inventario en cada paso.

In [ ]:
# Ejercicio 2



---

### 🔴 Ejercicio 3 — Semáforo *(difícil)*

Crea una clase `Semaforo` que:

- Tenga una property `color` que **solo acepte** los valores `"rojo"`, `"amarillo"` o `"verde"`
- Tenga una property de solo lectura `puede_pasar` que retorne `True` solo si el color es `"verde"`
- Tenga un método `siguiente()` que avance al próximo color en el ciclo: `verde → amarillo → rojo → verde → ...`
- Lleve internamente un contador `_cambios` (privado, sin property) que cuente cuántas veces cambió de color
- Tenga un método `informe()` que imprima el color actual, si se puede pasar y cuántos cambios lleva

**Prueba:** crea un semáforo en rojo, llama a `siguiente()` varias veces e imprime el informe en cada paso. Intenta asignar un color inválido como `"azul"`.

In [ ]:
# Ejercicio 3



---

### 🟣 Ejercicio 4 — Libre *(creativo)*

Piensa en un objeto del mundo real que tenga **reglas naturales** sobre sus atributos y modélalo con encapsulación.

Requisitos mínimos:
- Al menos **2 atributos privados** con sus properties
- Al menos **1 setter con validación**
- Al menos **1 property de solo lectura calculada**

Ideas: batería de celular, velocímetro de auto, nota de alumno, depósito de agua, reproductor de música...

In [ ]:
# Ejercicio 4

